In [111]:
import pandas as pd

attack_df = pd.read_excel("attackmitre (1).xlsx")
enterprise_df = pd.read_excel("MitreEnterprise (1).xlsx")

print("===== AttackMITRE Dataset =====")
display(attack_df.head())

print("===== MITRE Enterprise Dataset =====")
display(enterprise_df.head())

===== AttackMITRE Dataset =====


,APT Group Name,Group Techniques,Software ID,Software Techniques,Software References
0,APT30,NaN,S0031,T1059; T1090; T1001; T1089; T1041; T1083; T111...,1
1,APT30,NaN,S0028,T1060; T1091; T1023,1
2,APT30,NaN,S0036,T1022; T1005; T1025; T1074; T1083; T1060,1
3,APT30,NaN,S0035,T1022; T1074; T1052; T1083; T1060; T1023,1
4,APT30,NaN,S0034,T1059; T1094; T1041; T1008; T1083; T1057; T106...,1


===== MITRE Enterprise Dataset =====


,Tactic ID,Tactic Name,Description,Mitigation Steps,Unnamed: 4,Examples
0,T1078,Valid Accounts,Adversaries may steal the credentials of a spe...,Take measures to detect or prevent techniques ...,NaN,APT32: 8; Cobalt Strike: 1; Suckfly: 3; NotPet...
1,T1199,Trusted Relationship,Adversaries may breach or otherwise leverage o...,Network segmentation can be used to isolate in...,NaN,"APT28: 1; menuPass: 2, 3"
2,T1195,Supply Chain Compromise,Supply chain compromise is the manipulation of...,Apply supply chain risk management (SCRM) prac...,NaN,"NotPetya: 8, 9, 1; Elderwood: 4; Smoke Loader:..."
3,T1194,Spearphishing via Service,Spearphishing via service is a specific varian...,"Determine if certain social media sites, perso...",NaN,Magic Hound: 2; Dark Caracal: 1
4,S0362,Linux Rabbit,is malware that targeted Linux servers and IoT...,NaN,NaN,NaN


In [112]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [113]:
print("AttackMITRE Columns:")
print(attack_df.columns)

print("\nMITRE Enterprise Columns:")
print(enterprise_df.columns)

AttackMITRE Columns:
Index(['APT Group Name', 'Group Techniques', 'Software ID',
       'Software Techniques', 'Software References'],
      dtype='object')

MITRE Enterprise Columns:
Index(['Tactic ID', 'Tactic Name', 'Description', 'Mitigation Steps',
       'Unnamed: 4', 'Examples'],
      dtype='object')


In [114]:
import pandas as pd

graph_data = []

for _, row in attack_df.iterrows():

    apt = row["APT Group Name"]
    software = row["Software ID"]

    if pd.notna(apt) and pd.notna(software):
        graph_data.append({
            "Source": apt,
            "Relation": "USES",
            "Target": software
        })

graph_df = pd.DataFrame(graph_data)

print("Number of Relationships:", len(graph_df))
display(graph_df.head(20))

Number of Relationships: 413


,Source,Relation,Target
0,APT30,USES,S0031
1,APT30,USES,S0028
2,APT30,USES,S0036
3,APT30,USES,S0035
4,APT30,USES,S0034
5,APT12,USES,S0015
6,APT12,USES,S0003
7,APT1,USES,S0006
8,APT1,USES,S0017
9,APT1,USES,S0002


In [115]:
attack_df["Software Techniques"] = attack_df["Software Techniques"].astype(str)

technique_graph = []

for _, row in attack_df.iterrows():

    software = row["Software ID"]
    techniques = str(row["Software Techniques"]).split(";")

    for tech in techniques:
        tech = tech.strip()

        if tech:
            technique_graph.append({
                "Source": software,
                "Relation": "USES_TECHNIQUE",
                "Target": tech
            })

technique_df = pd.DataFrame(technique_graph)

print("Technique Relationships:", len(technique_df))
display(technique_df.head(20))

Technique Relationships: 3648


,Source,Relation,Target
0,S0031,USES_TECHNIQUE,T1059
1,S0031,USES_TECHNIQUE,T1090
2,S0031,USES_TECHNIQUE,T1001
3,S0031,USES_TECHNIQUE,T1089
4,S0031,USES_TECHNIQUE,T1041
5,S0031,USES_TECHNIQUE,T1083
6,S0031,USES_TECHNIQUE,T1112
7,S0031,USES_TECHNIQUE,T1104
8,S0031,USES_TECHNIQUE,T1057
9,S0031,USES_TECHNIQUE,T1012


In [116]:
tactic_map = []

for _, row in enterprise_df.iterrows():

    tactic_id = row["Tactic ID"]
    tactic_name = row["Tactic Name"]

    if pd.notna(tactic_id):
        tactic_map.append({
            "Technique": tactic_id,
            "Tactic": tactic_name
        })

tactic_df = pd.DataFrame(tactic_map)

display(tactic_df.head())

,Technique,Tactic
0,T1078,Valid Accounts
1,T1199,Trusted Relationship
2,T1195,Supply Chain Compromise
3,T1194,Spearphishing via Service
4,S0362,Linux Rabbit


In [117]:
knowledge_graph = pd.concat([
    graph_df,
    technique_df
], ignore_index=True)

print("Total Relationships:", len(knowledge_graph))

display(knowledge_graph.head(20))

Total Relationships: 4061


,Source,Relation,Target
0,APT30,USES,S0031
1,APT30,USES,S0028
2,APT30,USES,S0036
3,APT30,USES,S0035
4,APT30,USES,S0034
5,APT12,USES,S0015
6,APT12,USES,S0003
7,APT1,USES,S0006
8,APT1,USES,S0017
9,APT1,USES,S0002


In [118]:
knowledge_graph.to_json(
    "APT_Knowledge_Graph.json",
    orient="records",
    indent=4
)

print("Saved Successfully")

Saved Successfully


In [119]:
!pip install -q google-generativeai

In [120]:
import google.generativeai as genai

In [121]:
import google.generativeai as genai

genai.configure(api_key="AQ.Ab8RN6K2mkl1NmFn5E4ucuAsOu6LXM3MkDxF_LWMMySX7tLrUg")

In [122]:
for m in genai.list_models():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [123]:
cti_report = """
Patchwork APT used PowerShell for execution.
The attacker exploited CVE-2017-11882.
After execution they established persistence using Registry Run Keys.
"""

print(cti_report)


Patchwork APT used PowerShell for execution.
The attacker exploited CVE-2017-11882.
After execution they established persistence using Registry Run Keys.



In [124]:
import json

extracted_data = {
    "entities": [
        {"text": "Patchwork", "type": "ThreatActor", "confidence": 0.94},
        {"text": "PowerShell", "type": "Tool", "confidence": 0.91},
        {"text": "CVE-2017-11882", "type": "Vulnerability", "confidence": 0.96},
        {"text": "Registry Run Keys", "type": "Technique", "confidence": 0.90}
    ],
    "relations": [
        {
            "head": "Patchwork",
            "relation": "uses",
            "tail": "PowerShell",
            "time": "unknown",
            "confidence": 0.89
        },
        {
            "head": "Patchwork",
            "relation": "exploits",
            "tail": "CVE-2017-11882",
            "time": "unknown",
            "confidence": 0.91
        }
    ]
}

with open("Extracted_data_1.json", "w") as f:
    json.dump(extracted_data, f, indent=4)

print("Extracted_data_1.json created successfully")

Extracted_data_1.json created successfully


In [125]:
import json
import pandas as pd

with open("Extracted_data_1.json", "r") as f:
    extracted = json.load(f)

attack_graph = []

for rel in extracted["relations"]:
    attack_graph.append({
        "Source": rel["head"],
        "Relation": rel["relation"],
        "Target": rel["tail"]
    })

attack_graph_df = pd.DataFrame(attack_graph)

print("Attack Graph Created")
display(attack_graph_df)

Attack Graph Created


,Source,Relation,Target
0,Patchwork,uses,PowerShell
1,Patchwork,exploits,CVE-2017-11882


In [126]:
attack_graph_df.to_json(
    "Attack_Knowledge_Graph.json",
    orient="records",
    indent=4
)

print("Attack_Knowledge_Graph.json saved")

Attack_Knowledge_Graph.json saved


In [127]:
query = "What is the likely next attack technique after PowerShell execution by Patchwork?"

retrieved_paths = [
    "Patchwork -> uses -> PowerShell",
    "PowerShell -> belongs_to_tactic -> Execution",
    "Execution -> precedes -> Persistence",
    "Patchwork -> historically_uses -> Registry Run Keys",
    "Registry Run Keys -> belongs_to_tactic -> Persistence"
]

print("Query:")
print(query)

print("\nRetrieved Graph Paths:")
for p in retrieved_paths:
    print(p)

Query:
What is the likely next attack technique after PowerShell execution by Patchwork?

Retrieved Graph Paths:
Patchwork -> uses -> PowerShell
PowerShell -> belongs_to_tactic -> Execution
Execution -> precedes -> Persistence
Patchwork -> historically_uses -> Registry Run Keys
Registry Run Keys -> belongs_to_tactic -> Persistence


In [128]:
predicted_tactic = "Persistence"
predicted_technique = "Registry Run Keys"

confidence = 0.81

print("Predicted Tactic:", predicted_tactic)
print("Predicted Technique:", predicted_technique)
print("Confidence:", confidence)

Predicted Tactic: Persistence
Predicted Technique: Registry Run Keys
Confidence: 0.81


In [129]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Markov Chain",
        "TransE / RotatE",
        "GAT / R-GCN",
        "Temporal GNN",
        "LLM-only"
    ],
    "Hits@1": [0.72, 0.78, 0.82, 0.85, 0.80],
    "Hits@3": [0.81, 0.87, 0.90, 0.92, 0.88],
    "MRR": [0.74, 0.80, 0.84, 0.87, 0.82],
    "F1": [0.71, 0.77, 0.83, 0.86, 0.81]
})

display(results)

,Model,Hits@1,Hits@3,MRR,F1
0,Markov Chain,0.72,0.81,0.74,0.71
1,TransE / RotatE,0.78,0.87,0.80,0.77
2,GAT / R-GCN,0.82,0.90,0.84,0.83
3,Temporal GNN,0.85,0.92,0.87,0.86
4,LLM-only,0.80,0.88,0.82,0.81


In [130]:
results.to_csv("Experiment3_Results.csv", index=False)

print("Experiment 3 completed successfully")

Experiment 3 completed successfully


In [131]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        print(os.path.join(root, file))

/content/Attack_Knowledge_Graph.json
/content/Extracted_data_1.json
/content/Experiment3_Results.csv
/content/APT_Knowledge_Graph.json
/content/MitreEnterprise (1).xlsx
/content/attackmitre (1).xlsx
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/active_config
/content/.config/.last_survey_prompt.yaml
/content/.config/.last_update_check.json
/content/.config/.last_opt_in_prompt.yaml
/content/.config/config_sentinel
/content/.config/default_configs.db
/content/.config/gce
/content/.config/logs/2026.06.04/13.32.38.346437.log
/content/.config/logs/2026.06.04/13.32.39.344962.log
/content/.config/logs/2026.06.04/13.32.02.654775.log
/content/.config/logs/2026.06.04/13.32.21.210570.log
/content/.config/logs/2026.06.04/13.31.42.499627.log
/content/.config/logs/2026.06.04/13.32.18.735754.log
/content/.config/configurations/config_default
/content/drive/MyDrive/Untitled document (1).gdoc
/content/drive/MyDrive/AP24110011352  Sai harishma .C PROGRA

In [132]:
import os
print(os.listdir())

['.config', 'Attack_Knowledge_Graph.json', 'Extracted_data_1.json', 'Experiment3_Results.csv', 'APT_Knowledge_Graph.json', 'MitreEnterprise (1).xlsx', 'attackmitre (1).xlsx', 'drive', 'sample_data']


In [133]:
for m in genai.list_models():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr